# Pregões eletrônicos em São Paulo — PNCP
**Autor: Klayton Silva | Projeto de portfólio | Versão 1**

## Objetivo e perguntas
Analisar a distribuição dos registros por município da unidade do órgão, situação e data de publicação, além da qualidade e distribuição dos valores estimados.

**Recorte:** publicações de 01 a 07/08/2026, UF SP, modalidade Pregão Eletrônico. São 1.660 registros coletados em 166 páginas de 10 registros. Não há comparação entre modalidades nesta versão.

## Como usar
Execute as células na ordem. No Colab, envie `pncp_sp.sqlite` quando solicitado. Se o banco já estiver na mesma pasta, ele será aberto diretamente. Este notebook apenas lê o banco: não refaz a coleta nem substitui tabelas. As saídas salvas foram recalculadas sobre o banco fornecido.

## Origem e método
A coleta foi feita com Python e requests no endpoint `/v1/contratacoes/publicacao`, com `dataInicial=20260801`, `dataFinal=20260807`, `uf=SP`, `codigoModalidadeContratacao=6`, `tamanhoPagina=10` e paginação de 1 a 166. As páginas foram armazenadas em um dicionário, reunidas com Pandas e exportadas para SQLite. Foram conferidas páginas faltantes, duplicatas, datas e valores ausentes.

[Documentação oficial do PNCP](https://pncp.gov.br/api/consulta/swagger-ui/index.html). A coleta original é preservada em `historico_coleta.ipynb` como registro de aprendizagem; ela foi executada em etapas e não é um coletor automatizado para executar tudo uma vez. O JSON bruto não integra este pacote porque não foi enviado nesta revisão. A data exata de coleta não foi registrada no banco.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

# No Colab, o renderizador padrão permite interagir com os gráficos.
if 'google.colab' not in str(get_ipython()):
    pio.renderers.default = 'plotly_mimetype'

caminho = Path('pncp_sp.sqlite')
if not caminho.is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Coloque pncp_sp.sqlite na mesma pasta do notebook.')
    enviados = files.upload()
    candidatos = [nome for nome in enviados if nome.endswith('.sqlite')]
    if len(candidatos) != 1:
        raise ValueError('Envie somente um banco .sqlite.')
    caminho = Path(candidatos[0])

conexao = sqlite3.connect(caminho.resolve().as_uri() + '?mode=ro', uri=True)
print('Integridade:', conexao.execute('PRAGMA integrity_check').fetchone()[0])
df = pd.read_sql_query('SELECT * FROM contratacoes', conexao)
print('Registros:', len(df))


Saving pncp_sp.sqlite to pncp_sp.sqlite
Integridade: ok
Registros: 1660


## 1. Qualidade dos dados
Cada linha representa um registro de contratação. O identificador PNCP deve estar preenchido e ser único. Conferimos também o período e o recorte antes de analisar.


In [ ]:
datas = pd.to_datetime(df['data_publicacao'], errors='coerce')
checagens = pd.Series({
    'Registros': len(df),
    'IDs ausentes': int(df['id_contratacao'].isna().sum()),
    'IDs repetidos': int(df['id_contratacao'].duplicated().sum()),
    'Datas inválidas ou ausentes': int(datas.isna().sum()),
    'Fora do período': int(((datas < '2026-08-01') | (datas >= '2026-08-08')).sum()),
    'Fora da UF SP': int((df['uf'].fillna('') != 'SP').sum()),
    'Fora da modalidade': int((df['modalidade'].fillna('') != 'Pregão - Eletrônico').sum()),
    'Municípios ausentes': int(df['municipio'].isna().sum())
}, name='quantidade')
display(checagens.to_frame())
assert len(df) == 1660, 'O banco difere do recorte original.'
assert checagens.iloc[1:].eq(0).all(), 'Revise os problemas de qualidade acima.'
valores = df[['valor_estimado', 'valor_homologado']].apply(pd.to_numeric, errors='coerce')
resumo_qualidade = pd.DataFrame({
    'ausentes': valores.isna().sum(),
    'zeros': valores.eq(0).sum(),
    'positivos': valores.gt(0).sum(),
    'negativos': valores.lt(0).sum(),
    'preenchidos_nao_numericos': (df[valores.columns].notna() & valores.isna()).sum()
})
display(resumo_qualidade)


,quantidade
Registros,1660
IDs ausentes,0
IDs repetidos,0
Datas inválidas ou ausentes,0
Fora do período,0
Fora da UF SP,0
Fora da modalidade,0
Municípios ausentes,0


,ausentes,zeros,positivos,negativos,preenchidos_nao_numericos
valor_estimado,0,283,1377,0,0
valor_homologado,942,1,717,0,0


**Decisão:** preservar zeros e ausências, sem preenchimento automático. Há 283 estimativas iguais a zero e 942 valores homologados ausentes. A ausência não significa zero. Essas verificações não garantem que a API permaneceu imutável durante toda a paginação.

## 2. Distribuição por município
`GROUP BY` reúne registros do mesmo município; `COUNT(*)` conta as linhas. A participação usa os 1.660 registros como denominador.


In [ ]:
consulta_participacao = """
SELECT
    municipio,
    COUNT(*) AS quantidade,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM contratacoes),
        2
    ) AS percentual_total
FROM contratacoes
GROUP BY municipio
ORDER BY quantidade DESC, municipio ASC
LIMIT 10;
"""

participacao_municipios = pd.read_sql_query(
    consulta_participacao, conexao
)

display(participacao_municipios)

,municipio,quantidade,percentual_total
0,São Paulo,374,22.53
1,Campinas,42,2.53
2,Bauru,38,2.29
3,Ribeirão Preto,31,1.87
4,São José dos Campos,26,1.57
5,Santos,23,1.39
6,São José do Rio Preto,20,1.20
7,Mogi das Cruzes,19,1.14
8,Botucatu,18,1.08
9,Franca,18,1.08


In [ ]:
import plotly.express as px

fig = px.bar(
    participacao_municipios,
    x="quantidade",
    y="municipio",
    orientation="h",
    text="quantidade",
    title="Pregões eletrônicos: 10 municípios com mais registros",
    labels={
        "quantidade": "Quantidade de registros",
        "municipio": "Município da unidade"
    }
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=participacao_municipios["municipio"].tolist(),
    autorange="reversed"
)

fig.update_layout(
    height=550,
    margin=dict(l=10, r=20, t=80, b=50)
)
fig.update_layout(
    title=dict(
        text="Top 10 municípios<br>Pregões eletrônicos — SP<br>Publicações: 01 a 07/08/2026",
        font=dict(size=14),
        x=0
    ),
    xaxis_title="Registros",
    yaxis_title=None,
    font=dict(size=11),
    margin=dict(l=10, r=35, t=100, b=50)
)

fig.update_xaxes(tickfont=dict(size=10))
fig.show()

## Primeira conclusão: distribuição por município

No recorte de pregões eletrônicos publicados de 01 a 07/08/2026,
com unidades de órgãos localizadas em SP, foram coletados
1.660 registros, sem identificadores repetidos.

São Paulo apresentou a maior quantidade: 374 registros,
equivalentes a 22,53% do total. Campinas teve 42 registros
(2,53%) e Bauru, 38 (2,29%).

A localização corresponde ao município da unidade do órgão.
Os registros podem envolver órgãos municipais, estaduais
e federais, não apenas prefeituras.

Esse resultado mede quantidade de registros, não valores
gastos, e não permite concluir o motivo da concentração.

## 3. Situações e preenchimento do valor homologado
`COUNT(valor_homologado)` conta apenas valores preenchidos, incluindo zero. O percentual de ausência por situação usa o total do próprio grupo.

In [ ]:
consulta_situacoes = """
SELECT
    situacao,
    COUNT(*) AS quantidade,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM contratacoes),
        2
    ) AS percentual
FROM contratacoes
GROUP BY situacao
ORDER BY quantidade DESC, situacao ASC;
"""

resumo_situacoes = pd.read_sql_query(
    consulta_situacoes, conexao
)

display(resumo_situacoes)

,situacao,quantidade,percentual
0,Divulgada no PNCP,1588,95.66
1,Suspensa,44,2.65
2,Revogada,18,1.08
3,Anulada,10,0.60


In [ ]:
consulta_ausencia = """
SELECT
    situacao,
    COUNT(*) AS total,
    COUNT(*) - COUNT(valor_homologado) AS sem_valor,
    ROUND(
        100.0 * (COUNT(*) - COUNT(valor_homologado))
        / COUNT(*),
        2
    ) AS percentual_ausente
FROM contratacoes
GROUP BY situacao
ORDER BY percentual_ausente DESC, situacao ASC;
"""

ausencia_por_situacao = pd.read_sql_query(
    consulta_ausencia, conexao
)

display(ausencia_por_situacao)

,situacao,total,sem_valor,percentual_ausente
0,Anulada,10,10,100.00
1,Suspensa,44,42,95.45
2,Revogada,18,17,94.44
3,Divulgada no PNCP,1588,873,54.97


## Segunda conclusão: situações e valores ausentes

Dos 1.660 registros, 1.588 (95,66%) estavam na situação
“Divulgada no PNCP” no momento da coleta.

O percentual de ausência de valor homologado dentro de
cada situação foi:

- Anulada: 100% — 10 de 10 registros.
- Suspensa: 95,45% — 42 de 44 registros.
- Revogada: 94,44% — 17 de 18 registros.
- Divulgada no PNCP: 54,97% — 873 de 1.588 registros.

Essas proporções descrevem o preenchimento dos dados.
Não demonstram que a situação causou a ausência.

Os grupos têm tamanhos diferentes. “Divulgada no PNCP”
tem a menor proporção de ausência, mas a maior quantidade
absoluta de valores ausentes: 873.

Valores ausentes foram preservados, sem substituição
por zero. Valor homologado não representa valor pago.

## 4. Publicações diárias
O calendário completo é combinado à contagem com `left merge`. Aqui, preencher com zero indica nenhum registro encontrado para aquele dia no recorte; não representa um valor monetário desconhecido.

In [ ]:
consulta_publicacoes = """
SELECT
    SUBSTR(data_publicacao, 1, 10) AS dia,
    COUNT(*) AS quantidade
FROM contratacoes
GROUP BY SUBSTR(data_publicacao, 1, 10)
ORDER BY dia;
"""

publicacoes_por_dia = pd.read_sql_query(
    consulta_publicacoes, conexao
)

display(publicacoes_por_dia)
print("Total:", publicacoes_por_dia["quantidade"].sum())

,dia,quantidade
0,2026-08-02,1
1,2026-08-03,292
2,2026-08-04,353
3,2026-08-05,360
4,2026-08-06,326
5,2026-08-07,328


Total: 1660


In [ ]:
calendario = pd.DataFrame({
    "dia": pd.date_range("2026-08-01", "2026-08-07")
        .strftime("%Y-%m-%d")
})

publicacoes_semana = calendario.merge(
    publicacoes_por_dia,
    on="dia",
    how="left"
)

publicacoes_semana["quantidade"] = (
    publicacoes_semana["quantidade"].fillna(0).astype(int)
)

display(publicacoes_semana)

,dia,quantidade
0,2026-08-01,0
1,2026-08-02,1
2,2026-08-03,292
3,2026-08-04,353
4,2026-08-05,360
5,2026-08-06,326
6,2026-08-07,328


In [ ]:
import plotly.express as px

fig_diario = px.bar(
    publicacoes_semana,
    x="dia",
    y="quantidade",
    text="quantidade",
    labels={"dia": "Publicação", "quantidade": "Registros"}
)

fig_diario.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_diario.update_layout(
    title=dict(
        text="Publicações por dia<br>Pregões eletrônicos — SP<br>01 a 07/08/2026",
        font=dict(size=14)
    ),
    height=450,
    margin=dict(l=15, r=15, t=100, b=60),
    yaxis_range=[0, 420]
)

fig_diario.update_xaxes(
    type="category",
    tickvals=publicacoes_semana["dia"].tolist(),
    ticktext=["01/08", "02/08", "03/08", "04/08",
              "05/08", "06/08", "07/08"]
)

fig_diario.show()

### Conclusão diária

No recorte de pregões eletrônicos de SP, entre 01 e 07/08/2026, foram identificados 1.660 registros. O maior volume ocorreu em 05/08, com 360 registros.

Em 01/08, não foram encontrados registros no recorte coletado; em 02/08, foi encontrado apenas um. O calendário completo foi mantido no gráfico para mostrar também o dia sem registros.

Esta análise representa a quantidade de publicações, não os valores gastos ou pagos. O período de uma semana não permite afirmar uma tendência de longo prazo nem explicar as causas das diferenças entre os dias.

## 5. Distribuição dos valores estimados
Vamos descrever somente as estimativas **estritamente positivas** (1.377 registros). Os 283 zeros continuam no banco e nas verificações, mas ficam fora desta distribuição, pois sua causa não foi determinada.

A média soma os valores e divide pela quantidade. A mediana é o valor central da lista ordenada e sofre menos influência de valores muito altos. Valores são apresentados em reais; arredondamos apenas a exibição. Não comparamos totais estimados e homologados como economia, pois a cobertura dos campos é diferente.


In [ ]:
positivos = valores.loc[valores['valor_estimado'] > 0, 'valor_estimado']
resumo_estimados = pd.DataFrame({
    'medida': ['Quantidade', 'Mínimo (R$)', 'Mediana (R$)', 'Média (R$)', 'Máximo (R$)', 'Soma (R$)'],
    'valor': [positivos.count(), positivos.min(), positivos.median(), positivos.mean(), positivos.max(), positivos.sum()]
})
display(resumo_estimados.assign(valor=resumo_estimados['valor'].map(
    lambda x: f'{x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
)))
print(f'Participação da maior estimativa na soma: {positivos.max()/positivos.sum():.2%}')


,medida,valor
0,Quantidade,"1.377,00"
1,Mínimo (R$),"0,25"
2,Mediana (R$),"249.200,00"
3,Média (R$),"2.677.235,61"
4,Máximo (R$),"1.114.547.331,60"
5,Soma (R$),"3.686.553.437,90"


Participação da maior estimativa na soma: 30.23%


### Conclusão monetária
Entre as 1.377 estimativas positivas, a mediana foi **R$ 249.200,00** e a média, **R$ 2.677.235,61**. A maior estimativa foi **R$ 1.114.547.331,60**, equivalente a **30,23%** da soma de **R$ 3.686.553.437,90**.

A diferença entre média e mediana e a participação do maior registro mostram a influência de valores altos. Isso não comprova erro: seria necessário verificar o objeto e o documento de origem antes de classificar um registro como incorreto. A soma descreve estimativas registradas; não é despesa executada, pagamento nem compromisso de gasto.

## 6. Limitações e aprendizados
- Uma semana, uma UF e uma modalidade não representam todo o PNCP nem uma tendência de longo prazo.
- Município é a localização da unidade do órgão, que pode ser municipal, estadual ou federal; não necessariamente o local de entrega.
- Situação e valores refletem o estado dos registros na coleta, não necessariamente na publicação.
- Zero monetário tem causa desconhecida neste estudo. Ausências foram preservadas como `NULL` no SQLite.
- Valores estimados e homologados não representam valores efetivamente pagos.
- Contagens e IDs únicos ajudam a conferir a coleta, mas não comprovam um snapshot imutável da API.
- SQLite armazena os valores como `REAL`: esta análise é descritiva, não uma apuração contábil exata em centavos.

**O que aprendi:** coletar uma API paginada, conferir qualidade, usar SQLite e SQL, construir gráficos e escrever conclusões limitadas ao que os dados sustentam.

**Próximas melhorias:** automatizar checkpoints e tentativas de coleta, registrar data e metadados da extração e ampliar o período antes de investigar tendências.


In [ ]:
conexao.close()
print("Análise concluída. Banco preservado.")

Análise concluída. Banco preservado.
